In [ ]:
# run_ges_package_with_reg_and_bic_renamed.py
# ------------------------------------------------------------
# GES wrapper script (causal-learn)
# - Input: numeric-only CSV
# - Preprocess: drop targets + numeric-only + NaN median + centering
# - Learn: GES (structure)
# - Compute:
#     (A) Δ local BIC score per directed edge  -> "GES_BIC"
#     (B) OLS coefficients on fixed structure  -> "GES"
# - Enforce DAG: cycle-break on |score| (min_abs_score)
# - Save:
#     - edges_GES.csv, adj_GES_weight.csv, graph_GES.(gexf/graphml)
#     - edges_GES_BIC.csv, adj_GES_BIC_score.csv, graph_GES_BIC.(gexf/graphml)
#     - graph_GES_with_GES_BIC.json (both weight+score)
#
# Requirements:
#   pip install numpy pandas causal-learn networkx
# ------------------------------------------------------------

import os
import json
import warnings
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =========================
# Config
# =========================
DATA_PATH = "./training_data_standardization.csv"
OUT_BASE = "./dag_out/GES"
RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# GES score config (요구사항: 문자열로 전달)
GES_SCORE_FUNC = "local_score_BIC"

# cycle break: |score| 가장 작은 edge 제거
CYCLE_BREAK_STRATEGY = "min_abs_score"

# OLS weight estimation config (✅ GES 구조 고정 후 계수 추정)
USE_STANDARDIZE_FOR_OLS = True   # NOTEARS/GOLEM과 비교용으로 권장
OLS_RCOND = None                # np.linalg.lstsq rcond

# "0 edge" 제거 기준 (저장/그래프에서 제외)
EPS_EDGE = 1e-12  # 진짜 0만 제거하려면 0.0, 사실상 0도 제거하려면 1e-6 등


# =========================
# Data
# =========================
def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    if drop_target_candidates:
        drop_cols = [c for c in df.columns if c in TARGET_CANDIDATES]
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    # numeric only
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    # NaN -> median
    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)

    # (중요) GES score 계산의 일관성을 위해 센터링
    X = X - X.mean(axis=0, keepdims=True)

    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names


# =========================
# Graph helpers
# =========================
def extract_directed_edges_from_causallearn_graph(G: Any, d: int) -> List[Tuple[int, int]]:
    """
    causal-learn Graph 객체에서 i->j 방향 간선을 추출.
    PDAG/CPDAG의 경우 방향 확정된 i->j만 추출.
    """
    mat = getattr(G, "graph", None)
    if mat is None:
        raise ValueError("Cannot find adjacency matrix 'G.graph' in causal-learn Graph object.")

    mat = np.asarray(mat)
    if mat.shape != (d, d):
        raise ValueError(f"G.graph shape mismatch: {mat.shape} vs d={d}")

    edges: List[Tuple[int, int]] = []

    # Case A: i->j : mat[i,j]==1 and mat[j,i]==-1
    for i in range(d):
        for j in range(d):
            if i != j and mat[i, j] == 1 and mat[j, i] == -1:
                edges.append((i, j))
    if edges:
        return list(dict.fromkeys(edges))

    # Case B: i->j : mat[i,j]==-1 and mat[j,i]==1
    for i in range(d):
        for j in range(d):
            if i != j and mat[i, j] == -1 and mat[j, i] == 1:
                edges.append((i, j))
    if edges:
        return list(dict.fromkeys(edges))

    # Case C: mat[i,j]!=0 and mat[j,i]==0 를 i->j로 (fallback)
    for i in range(d):
        for j in range(d):
            if i != j and mat[i, j] != 0 and mat[j, i] == 0:
                edges.append((i, j))

    return list(dict.fromkeys(edges))


def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    """
    cycle이 있으면, cycle에 포함된 edge 중 |W|가 가장 작은 edge 제거.
    """
    W2 = W.copy()

    def build_graph(Wm):
        adj = {i: [] for i in range(Wm.shape[0])}
        for i in range(Wm.shape[0]):
            for j in range(Wm.shape[1]):
                if i != j and abs(Wm[i, j]) > 0:
                    adj[i].append(j)
        return adj

    def find_cycle_edges(adj):
        d = len(adj)
        color = [0] * d
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in adj[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    while True:
        cyc = find_cycle_edges(build_graph(W2))
        if cyc is None:
            break

        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0

    return W2


# =========================
# BIC score wrapper (version compatibility)
# =========================
def make_local_bic_scorer(X: np.ndarray):
    """
    causal-learn 버전 차로 local_score_BIC 호출 형태가 달라도 scorer(i, PAi)로 통일.
    """
    try:
        from causallearn.score.LocalScoreFunction import local_score_BIC
    except Exception as e:
        raise ImportError(
            "local_score_BIC를 찾을 수 없습니다. causal-learn 설치가 필요합니다.\n"
            "pip install causal-learn\n"
            f"Original error: {e}"
        )

    # 1) factory 형태 시도
    try:
        maybe = local_score_BIC(X)
        if callable(maybe):
            def scorer(i: int, PAi: List[int]) -> float:
                return float(maybe(i, PAi))
            return scorer
    except TypeError:
        pass
    except Exception:
        pass

    # 2) direct call 형태
    def scorer(i: int, PAi: List[int]) -> float:
        return float(local_score_BIC(X, i, PAi))
    return scorer


# =========================
# GES Δ local BIC score (edge importance)
# =========================
def compute_delta_scores_for_edges(
    X: np.ndarray,
    edges: List[Tuple[int, int]],
    d: int
) -> np.ndarray:
    """
    Δscore(i->j) = local_score(j | Pa_j ∪ {i}) - local_score(j | Pa_j)
    (Pa_j는 최종 그래프에서의 부모 집합에서 i 제외)
    """
    scorer = make_local_bic_scorer(X)

    parents_of: Dict[int, List[int]] = {j: [] for j in range(d)}
    for i, j in edges:
        parents_of[j].append(i)

    W_score = np.zeros((d, d), dtype=float)

    for i, j in edges:
        pa_wo_i = sorted([p for p in parents_of[j] if p != i])
        s0 = scorer(j, pa_wo_i)
        s1 = scorer(j, sorted(pa_wo_i + [i]))
        W_score[i, j] = float(s1 - s0)

    return W_score


# =========================
# OLS coefficients on fixed GES structure (feature weight)
# =========================
def standardize(X: np.ndarray) -> np.ndarray:
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True)
    sd = np.where(sd < 1e-12, 1.0, sd)
    return (X - mu) / sd


def compute_ols_weights_for_edges(
    X: np.ndarray,
    edges: List[Tuple[int, int]],
    d: int,
    standardize_for_ols: bool = True
) -> np.ndarray:
    """
    GES로 얻은 구조(부모집합)를 고정하고,
    각 노드 j에 대해: X_j ~ X_{Pa(j)} 선형회귀(절편 없음)로 계수 추정.
    반환 W_beta (d x d): i->j일 때 W_beta[i,j] = beta_{i->j}
    """
    if len(edges) == 0:
        return np.zeros((d, d), dtype=float)

    Xr = standardize(X) if standardize_for_ols else X.copy()

    parents_of: Dict[int, List[int]] = {j: [] for j in range(d)}
    for i, j in edges:
        parents_of[j].append(i)

    W_beta = np.zeros((d, d), dtype=float)

    for j in range(d):
        pa = sorted(list(dict.fromkeys(parents_of[j])))
        if len(pa) == 0:
            continue

        y = Xr[:, j]
        Xp = Xr[:, pa]

        # least squares (no intercept; X already centered/standardized if enabled)
        coef, *_ = np.linalg.lstsq(Xp, y, rcond=OLS_RCOND)
        coef = coef.astype(float)

        for k, i in enumerate(pa):
            W_beta[i, j] = float(coef[k])

    return W_beta


# =========================
# Save artifacts (renamed)
# =========================
def save_artifacts_renamed(
    W_score: np.ndarray,
    W_beta: np.ndarray,
    col_names: List[str],
    out_dir: str,
    eps_edge: float,
    base_name_score: str = "GES_BIC",
    base_name_reg: str = "GES",
) -> None:
    """
    원하는 이름 규칙:
    - base_name_score (예: "GES_BIC"): score 그래프/행렬/edge 테이블
    - base_name_reg   (예: "GES")    : regression(beta) 그래프/행렬/edge 테이블

    Saved:
    - edges_{base_name_reg}.csv              (weight)
    - adj_{base_name_reg}_weight.csv
    - graph_{base_name_reg}.gexf/.graphml    (weight on edges)

    - edges_{base_name_score}.csv            (score)
    - adj_{base_name_score}_score.csv
    - graph_{base_name_score}.gexf/.graphml  (score on edges)

    - graph_{base_name_reg}_with_{base_name_score}.json  (both)
    """
    import networkx as nx

    os.makedirs(out_dir, exist_ok=True)
    d = len(col_names)

    # ---- score edge table ----
    rows_score = []
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            sc = float(W_score[i, j])
            if abs(sc) <= eps_edge:
                continue
            rows_score.append([col_names[i], col_names[j], sc])

    edge_score_df = pd.DataFrame(rows_score, columns=["source", "target", "score"])
    edge_score_path = os.path.join(out_dir, f"edges_{base_name_score}.csv")
    edge_score_df.to_csv(edge_score_path, index=False)

    adj_score = pd.DataFrame(W_score, index=col_names, columns=col_names)
    score_adj_path = os.path.join(out_dir, f"adj_{base_name_score}_score.csv")
    adj_score.to_csv(score_adj_path)

    G_score = nx.DiGraph()
    G_score.add_nodes_from(col_names)
    for _, r in edge_score_df.iterrows():
        G_score.add_edge(r["source"], r["target"], score=float(r["score"]))

    gexf_score_path = os.path.join(out_dir, f"graph_{base_name_score}.gexf")
    graphml_score_path = os.path.join(out_dir, f"graph_{base_name_score}.graphml")
    nx.write_gexf(G_score, gexf_score_path)
    nx.write_graphml(G_score, graphml_score_path)

    # ---- regression edge table ----
    rows_reg = []
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            beta = float(W_beta[i, j])
            if abs(beta) <= eps_edge:
                continue
            rows_reg.append([col_names[i], col_names[j], beta])

    edge_reg_df = pd.DataFrame(rows_reg, columns=["source", "target", "weight"])
    edge_reg_path = os.path.join(out_dir, f"edges_{base_name_reg}.csv")
    edge_reg_df.to_csv(edge_reg_path, index=False)

    adj_beta = pd.DataFrame(W_beta, index=col_names, columns=col_names)
    beta_adj_path = os.path.join(out_dir, f"adj_{base_name_reg}_weight.csv")
    adj_beta.to_csv(beta_adj_path)

    G_reg = nx.DiGraph()
    G_reg.add_nodes_from(col_names)
    for _, r in edge_reg_df.iterrows():
        G_reg.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    gexf_reg_path = os.path.join(out_dir, f"graph_{base_name_reg}.gexf")
    graphml_reg_path = os.path.join(out_dir, f"graph_{base_name_reg}.graphml")
    nx.write_gexf(G_reg, gexf_reg_path)
    nx.write_graphml(G_reg, graphml_reg_path)

    # ---- JSON (both) ----
    json_path = os.path.join(out_dir, f"graph_{base_name_reg}_with_{base_name_score}.json")

    edge_map: Dict[Tuple[str, str], Dict[str, float]] = {}
    for _, r in edge_reg_df.iterrows():
        key = (str(r["source"]), str(r["target"]))
        edge_map[key] = {"weight": float(r["weight"]), "score": 0.0}

    for _, r in edge_score_df.iterrows():
        key = (str(r["source"]), str(r["target"]))
        if key not in edge_map:
            edge_map[key] = {"weight": 0.0, "score": float(r["score"])}
        else:
            edge_map[key]["score"] = float(r["score"])

    nodes = [{"id": n} for n in col_names]
    jedges = [
        {"source": s, "target": t, "weight": v["weight"], "score": v["score"]}
        for (s, t), v in edge_map.items()
    ]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"nodes": nodes, "edges": jedges}, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] renamed outputs")
    print(f"  - (reg/weights) {base_name_reg}")
    print(f"    * {edge_reg_path} (n_edges={len(edge_reg_df)})")
    print(f"    * {beta_adj_path}")
    print(f"    * {graphml_reg_path}")
    print(f"    * {gexf_reg_path}")
    print(f"  - (score/ΔBIC)  {base_name_score}")
    print(f"    * {edge_score_path} (n_edges={len(edge_score_df)})")
    print(f"    * {score_adj_path}")
    print(f"    * {graphml_score_path}")
    print(f"    * {gexf_score_path}")
    print(f"  - (json/both)   {json_path}")
    print(f"  - filtered: abs(value) > {eps_edge}")


# =========================
# Main
# =========================
def main():
    _, X, col_names = load_numeric_X(
        DATA_PATH,
        drop_target_candidates=True,
        max_features=MAX_FEATURES,
        random_state=RANDOM_STATE
    )

    try:
        from causallearn.search.ScoreBased.GES import ges
    except Exception as e:
        raise ImportError(
            "GES 실행을 위해 causal-learn이 필요합니다.\n"
            "pip install causal-learn\n"
            f"Original error: {e}"
        )

    # 1) Run GES
    Record = ges(X, score_func=GES_SCORE_FUNC)

    # 2) Get Graph object
    if isinstance(Record, dict) and "G" in Record:
        G = Record["G"]
    else:
        G = getattr(Record, "G", None)
        if G is None:
            raise ValueError("Cannot access learned graph from GES result. Expected Record['G'] or Record.G")

    d = len(col_names)

    # 3) Extract directed edges (directed-only)
    edges_ij = extract_directed_edges_from_causallearn_graph(G, d)
    print(f"[INFO] directed edges from GES: {len(edges_ij)}")

    # 4) Compute Δ local BIC score per edge
    W_score = compute_delta_scores_for_edges(X, edges_ij, d)

    # 5) Enforce DAG using Δscore (remove smallest |score|)
    if CYCLE_BREAK_STRATEGY == "min_abs_score":
        W_score = break_cycles_by_removing_small_edges(W_score)

        # remove edges that were zeroed out by cycle-break
        edges_ij = [(i, j) for (i, j) in edges_ij if abs(W_score[i, j]) > 0]
        print(f"[INFO] edges after cycle-break: {len(edges_ij)}")
    else:
        raise ValueError(f"Unknown CYCLE_BREAK_STRATEGY: {CYCLE_BREAK_STRATEGY}")

    # 6) Fit OLS coefficients on fixed structure
    W_beta = compute_ols_weights_for_edges(
        X=X,
        edges=edges_ij,
        d=d,
        standardize_for_ols=USE_STANDARDIZE_FOR_OLS
    )

    # Optional cleanup: if beta is ~0, also zero out score to keep outputs consistent
    for (i, j) in edges_ij:
        if abs(W_beta[i, j]) <= EPS_EDGE:
            W_beta[i, j] = 0.0
            W_score[i, j] = 0.0

    # 7) Save renamed artifacts:
    #    - beta graph -> "GES"
    #    - score graph -> "GES_BIC"
    save_artifacts_renamed(
        W_score=W_score,
        W_beta=W_beta,
        col_names=col_names,
        out_dir=OUT_BASE,
        eps_edge=EPS_EDGE,
        base_name_score="GES_BIC",
        base_name_reg="GES",
    )

    print("[DONE] GES outputs saved (GES=reg weights, GES_BIC=ΔBIC scores).")


if __name__ == "__main__":
    main()


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 13)
[INFO] directed edges from GES: 35
[INFO] edges after cycle-break: 35
[SAVE] renamed outputs
  - (reg/weights) GES
    * ./dag_out/GES\edges_GES.csv (n_edges=35)
    * ./dag_out/GES\adj_GES_weight.csv
    * ./dag_out/GES\graph_GES.graphml
    * ./dag_out/GES\graph_GES.gexf
  - (score/ΔBIC)  GES_BIC
    * ./dag_out/GES\edges_GES_BIC.csv (n_edges=35)
    * ./dag_out/GES\adj_GES_BIC_score.csv
    * ./dag_out/GES\graph_GES_BIC.graphml
    * ./dag_out/GES\graph_GES_BIC.gexf
  - (json/both)   ./dag_out/GES\graph_GES_with_GES_BIC.json
  - filtered: abs(value) > 1e-12
[DONE] GES outputs saved (GES=reg weights, GES_BIC=ΔBIC scores).
